# Nested sampling on the same four parameters as `Actress_MCMC_Timing.ipynb`

Bare-bones counterpart to the emcee notebook. Same actress setup, same ld files, same
truth, same noise levels, same `RANDOM_SEED`, so the synthetic data vector fitted here is
*identical* to the one fitted there and the two posteriors are directly comparable.

**Sampled parameters** (4): `inc` [deg], `v_eq` [m/s], `lat_centre` [deg], `ff`.

**Data vector** (41): total flux excess (1) + disc-integrated line profile (`N_PROFILE_BINS`),
from one `rotate_lc(N=1, returndisc=True)` call.

---

## Two deliberate differences from the emcee notebook

**1. The prior on `ff` is tighter, and this is a correction, not a preference.**

`band_edges_from_ff` returns `lat_centre +/- arcsin(ff / cos(lat_centre))`. Requiring only
`ff < cos(lat_centre)` (the emcee notebook's condition) is enough for the arcsin to exist,
but NOT enough to keep the band inside the sphere: `lat_centre + h <= 90` requires

    ff <= cos(lat_centre) * sin(90 - lat_centre) = cos^2(lat_centre)

In the gap `cos^2(lat) < ff < cos(lat)` the upper edge exceeds 90 deg. `simulator.py:403`
passes `(upper+90)*pi/180 > pi` to `hp.query_strip`, which does not raise: it works in
`z = cos(theta)`, so it silently returns the band reflected about 90 deg. That band has
the *same area* (Archimedes), which is why `getfill` looks correct, but it sits at the
wrong latitudes. Verified directly: `query_strip(8.70, 131.30)` and
`query_strip(8.70, 48.70)` return identical pixel sets.

So this notebook uses `ff < cos^2(lat_centre)`, and `band_edges_from_ff` raises if an edge
leaves +/-90 deg rather than letting healpy reflect it.

**2. No rejection inside the likelihood.**

Nested sampling needs a prior *transform* (unit cube -> parameters), not a prior density.
The `ff < cos^2(lat)` constraint is folded into the transform exactly, by inverse-CDF on
`lat_centre` followed by a conditional draw on `ff`, so every sample is valid by
construction and no likelihood call is wasted. Doing it the lazy way (uniform box +
`-inf` for violations) would change nothing about the posterior but would burn evaluations.

The resulting prior is *joint uniform* over the allowed `(lat_centre, ff)` region, which is
what the emcee `log_prior` rejection scheme implies. The alternative reading -- uniform in
`lat_centre`, then uniform in `ff` on whatever slice is allowed at that latitude -- is a
genuinely different prior. Cell "prior transform check" demonstrates the implemented one
matches rejection sampling.

Everything else -- forward model, noise, likelihood -- is copied verbatim.

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import os
import time

import numpy as np
import matplotlib.pyplot as plt

import dynesty
from dynesty import utils as dyfunc
import corner

import actress as ac   # REFRESH BEFORE USE!

print('dynesty', dynesty.__version__)
print('actress', ac.__file__)
print('cores visible to this process:', len(os.sched_getaffinity(0)))

In [ ]:
# =============================================================================
# Configuration
# =============================================================================
# The physics block below is copied value-for-value from Actress_MCMC_Timing.ipynb.
# Changing anything in it breaks the comparability of the two posteriors.

RANDOM_SEED = 20260807          # seeds the synthetic noise AND the sampler

# --- limb-darkening inputs ---
HD_LD_FILE  = 'test_phot.txt'       # quiet photosphere: columns [wavelength /m, I0, a, b]
FAC_LD_FILE = 'test_fac_200G.txt'   # faculae, same column layout
LD_LAW      = 'power2'

# --- model resolution ---
RES_3D = 6      # healpix resolution, nside = 2**RES_3D
RES_2D = 200    # side length of the projected disc image, in pixels

# --- spectral binning of the disc-integrated line profile ---
LINE_WAVELENGTH_ANG = 4375.0    # rest wavelength of the line being modelled [Angstrom]
N_PROFILE_BINS = 40             # number of Doppler bins across the profile
PROFILE_HALF_WIDTH_ANG = 0.09   # profile spans +/- this about line centre [Angstrom]

# --- the truth: the facular distribution we will try to recover ---
TRUE_INC        = 75.0     # stellar inclination [deg], 90 = equator-on
TRUE_V_EQ       = 4000.0   # equatorial rotation velocity [m/s]
TRUE_LAT_CENTRE = 25.0     # centre latitude of the facular band [deg]
TRUE_FF         = 0.12     # surface filling factor of the band [fraction of 4pi]

# --- synthetic noise levels ---
SIGMA_FLUX_REL    = 1.0e-4   # fractional uncertainty on the total flux excess
SIGMA_PROFILE_REL = 1.0e-3   # per-bin uncertainty, as a fraction of the profile peak

# --- prior bounds (same as the emcee notebook; see header for the ff constraint) ---
PRIOR_INC_DEG    = (0.0, 90.0)       # sampled uniform in cos(inc)
PRIOR_V_EQ_MS    = (0.0, 20000.0)
PRIOR_LAT_DEG    = (0.0, 80.0)       # single band in the northern hemisphere
PRIOR_FF         = (1.0e-3, 0.5)

# --- nested sampling settings ---
N_LIVE = 500            # live points. Cost scales ~linearly with this; posterior
                        # resolution in the tails improves with it.
SAMPLE_METHOD = 'rslice'  # random slice sampling. Chosen over the default 'auto'/'unif'
                          # because v_eq and inc lie on a CURVED ridge (they enter the
                          # Doppler map only as v_eq*sin(inc), simulator.py:500), and
                          # ellipsoidal rejection sampling tracks curvature poorly.
DLOGZ = 0.1             # stopping criterion: remaining evidence estimate
N_JOBS_INNER = 1        # joblib threads INSIDE rotate_lc. Left at 1 to match the
                        # emcee timing; this notebook runs the sampler serially.

TRUTH = np.array([TRUE_INC, TRUE_V_EQ, TRUE_LAT_CENTRE, TRUE_FF])
LABELS = ['inc [deg]', 'v_eq [m/s]', 'lat_centre [deg]', 'ff']
N_DIM = len(TRUTH)

In [ ]:
# =============================================================================
# Limb-darkening coefficients
# =============================================================================
# Single spectral line: take the ld row closest to LINE_WAVELENGTH_ANG and hold the
# coefficients FIXED. They are not sampled, so any error in them is absorbed as a
# systematic in ff rather than propagated into the error bars.

hd_ld_table  = np.loadtxt(HD_LD_FILE)
fac_ld_table = np.loadtxt(FAC_LD_FILE)

if hd_ld_table.shape != fac_ld_table.shape:
    raise ValueError('photosphere and facula ld tables have different shapes: '
                     '{} vs {}'.format(hd_ld_table.shape, fac_ld_table.shape))

# column 0 of the ld files is wavelength in METRES; our line is quoted in Angstroms
wavelengths_ang = hd_ld_table[:, 0] * 1.0e10
ld_row = int(np.argmin(np.abs(wavelengths_ang - LINE_WAVELENGTH_ANG)))

PHOT_COEFFS = list(hd_ld_table[ld_row, 1:4])    # [I0, a, b] for the power2 law
FAC_COEFFS  = list(fac_ld_table[ld_row, 1:4])
SPOT_COEFFS = [1.0e7, 0.3, 0.2]                 # unused: every run below is mode='faconly'


def power2_ld(mu, I0, a, b):
    """Power-2 limb darkening, matching Transitsim.ld_law's 'power2' branch."""
    return I0 * (1.0 - (a * (1.0 - (mu ** b))))


if LD_LAW != 'power2':
    raise NotImplementedError("this notebook hard-codes the power2 law; LD_LAW is '{}'".format(LD_LAW))

print('requested line   : {:.4f} A'.format(LINE_WAVELENGTH_ANG))
print('nearest ld row   : {} at {:.4f} A'.format(ld_row, wavelengths_ang[ld_row]))
print('photosphere [I0, a, b]:', PHOT_COEFFS)
print('faculae     [I0, a, b]:', FAC_COEFFS)

In [ ]:
# =============================================================================
# Band geometry: (centre latitude, filling factor) -> band edges
# =============================================================================
# Archimedes' hat-box theorem gives the area fraction of a spherical zone as
#
#     f = (sin(lat_hi) - sin(lat_lo)) / 2
#
# For a band of half-width h centred on lat_c this is f = cos(lat_c) * sin(h), so
#
#     h = arcsin( f / cos(lat_c) )
#
# TWO conditions must hold, not one:
#   (a) f <= cos(lat_c)      for the arcsin to exist at all
#   (b) lat_c + h <= 90      for the band to fit inside the sphere, which reduces
#                            to f <= cos^2(lat_c)
#
# (b) is strictly tighter than (a) and is the one that matters. Violating it does NOT
# raise inside healpy -- hp.query_strip works in z = cos(theta) and silently returns the
# band reflected about 90 deg, same area, wrong latitudes. See the header.

MAX_FF_AT_LAT = lambda lat_deg: np.cos(np.radians(lat_deg)) ** 2


def band_edges_from_ff(lat_centre_deg, ff):
    """
    Latitude edges [deg] of the band centred on lat_centre_deg whose surface
    filling factor is ff. Returns (lat_low, lat_high).

    Raises ValueError if the band cannot be realised at that centre latitude,
    including the pole-crossing case that healpy would otherwise absorb silently.
    """
    cos_lat_centre = np.cos(np.radians(lat_centre_deg))

    if ff > cos_lat_centre:
        raise ValueError(
            'ff={:.4f} is unreachable at lat_centre={:.2f} deg (max is cos(lat) = {:.4f})'
            .format(ff, lat_centre_deg, cos_lat_centre))

    half_width_deg = np.degrees(np.arcsin(ff / cos_lat_centre))

    lat_low = lat_centre_deg - half_width_deg
    lat_high = lat_centre_deg + half_width_deg

    # The loud version of the silent healpy reflection.
    if lat_low < -90.0 or lat_high > 90.0:
        raise ValueError(
            'band {:.2f} to {:.2f} deg leaves +/-90 for lat_centre={:.2f}, ff={:.4f}; '
            'the geometric limit here is cos^2(lat) = {:.4f}. healpy would silently '
            'reflect this band about 90 deg instead of failing.'
            .format(lat_low, lat_high, lat_centre_deg, ff, MAX_FF_AT_LAT(lat_centre_deg)))

    return lat_low, lat_high


# round-trip against actress's own pixel count, at the truth
_lo, _hi = band_edges_from_ff(TRUE_LAT_CENTRE, TRUE_FF)
_sim = ac.Simulator(faculae=[], spots=[], fac_strips=[], spot_strips=[])
_sim.setxsize(RES_2D)
_sim.setresolution(RES_3D)
_sim.setld({'phot': PHOT_COEFFS, 'spot': SPOT_COEFFS, 'fac': FAC_COEFFS, 'func': power2_ld})
_sim.addstrip(lower=_lo, upper=_hi, feature='fac')

print('band edges for lat_centre={} deg, ff={}: {:.4f} to {:.4f} deg'
      .format(TRUE_LAT_CENTRE, TRUE_FF, _lo, _hi))
print('ff requested (analytic) : {:.5f}'.format(TRUE_FF))
print('ff realised  (healpix)  : {:.5f}'.format(_sim.getfill('fac')))
print('difference is healpix pixelisation at nside={}'.format(2 ** RES_3D))
print()
print('geometric ff ceiling at this latitude: cos^2({}) = {:.4f}'
      .format(TRUE_LAT_CENTRE, MAX_FF_AT_LAT(TRUE_LAT_CENTRE)))

In [ ]:
# =============================================================================
# Forward model: one disc snapshot -> (total flux excess, line profile)
# =============================================================================
# Copied verbatim from Actress_MCMC_Timing.ipynb so the two notebooks fit the same model.

PROFILE_BIN_EDGES = np.linspace(-PROFILE_HALF_WIDTH_ANG, PROFILE_HALF_WIDTH_ANG,
                                N_PROFILE_BINS + 1)
PROFILE_BIN_CENTRES = 0.5 * (PROFILE_BIN_EDGES[:-1] + PROFILE_BIN_EDGES[1:])


def run_actress(inc_deg, v_eq_ms, lat_centre_deg, ff, n_phase=1,
                res_2d=RES_2D, res_3d=RES_3D, n_jobs=N_JOBS_INNER, mode='faconly'):
    """
    Build the star, project it, and return (discs, shift_map).

    discs      : (n_phase, res_2d, res_2d) intensity maps, 0 outside the stellar disc
    shift_map  : (res_2d, res_2d) Doppler shift per pixel [Angstrom], static
    """
    lat_low, lat_high = band_edges_from_ff(lat_centre_deg, ff)

    # explicit empty lists: Simulator's [] defaults are shared between instances
    sim = ac.Simulator(faculae=[], spots=[], fac_strips=[], spot_strips=[])
    sim.setxsize(res_2d)
    sim.setresolution(res_3d)
    sim.setld({'phot': PHOT_COEFFS, 'spot': SPOT_COEFFS,
               'fac': FAC_COEFFS, 'func': power2_ld})
    sim.addstrip(lower=lat_low, upper=lat_high, feature='fac')

    discs, shift_map = sim.rotate_lc(v_eq=v_eq_ms, inc=inc_deg, N=n_phase,
                                     mode=mode, wavelength=LINE_WAVELENGTH_ANG,
                                     returndisc=True, njobs=n_jobs)

    return np.asarray(discs), shift_map


def line_profile_from_disc(disc, shift_map):
    """
    Disc-integrated line profile of ONE snapshot, by summing pixel intensities
    into Doppler bins. Returns (N_PROFILE_BINS,), unnormalised.

    Pixels whose shift falls outside PROFILE_BIN_EDGES ARE dropped, which would
    silently lose flux if the window were too narrow; the check below makes that loud.
    """
    bin_index = np.digitize(shift_map.ravel(), PROFILE_BIN_EDGES) - 1
    in_window = (bin_index >= 0) & (bin_index < N_PROFILE_BINS)

    flat = disc.ravel()
    profile = np.bincount(bin_index[in_window], weights=flat[in_window],
                          minlength=N_PROFILE_BINS)

    total = np.nansum(flat)
    lost = total - profile.sum()
    if lost > 1.0e-6 * total:
        raise ValueError(
            '{:.3%} of the disc flux falls outside the profile window. '
            'Increase PROFILE_HALF_WIDTH_ANG.'.format(lost / total))

    return profile


# --- flux of the featureless star, used to normalise the total flux ---
# A featureless star is uniform, so its projected disc is identical at every rotation
# angle and inclination: a single scalar, computed once rather than in the likelihood.
_quiet_discs, _ = run_actress(90.0, 0.0, TRUE_LAT_CENTRE, TRUE_FF, mode='quiet')
QUIET_FLUX = float(np.nanmean(_quiet_discs[0]))
print('featureless-star disc-mean flux: {:.6f}'.format(QUIET_FLUX))


def forward_model(theta, res_2d=RES_2D, res_3d=RES_3D, n_jobs=N_JOBS_INNER):
    """
    Map parameters -> the two observables of a single disc snapshot.

    theta = (inc [deg], v_eq [m/s], lat_centre [deg], ff)

    Returns
    -------
    flux_excess : scalar, disc-mean flux divided by the featureless-star flux.
                  ABSOLUTE: using it as data assumes an absolutely calibrated flux
                  or an independent quiet reference.
    profile     : (N_PROFILE_BINS,) line profile normalised to unit sum, i.e. SHAPE
                  only, deliberately free of the flux scale.
    """
    inc_deg, v_eq_ms, lat_centre_deg, ff = theta

    discs, shift_map = run_actress(inc_deg, v_eq_ms, lat_centre_deg, ff,
                                   res_2d=res_2d, res_3d=res_3d, n_jobs=n_jobs)

    flux_excess = float(np.nanmean(discs[0])) / QUIET_FLUX

    profile = line_profile_from_disc(discs[0], shift_map)
    profile = profile / profile.sum()

    return flux_excess, profile


_t0 = time.time()
_flux, _prof = forward_model(TRUTH)
T_FORWARD = time.time() - _t0
print('one forward_model call: {:.4f} s'.format(T_FORWARD))
print('flux excess           : {:.6f}'.format(_flux))
print('data vector           : 1 + {} = {} numbers, for {} parameters'
      .format(N_PROFILE_BINS, 1 + N_PROFILE_BINS, N_DIM))

In [ ]:
# =============================================================================
# Synthetic data: the single snapshot we will try to invert
# =============================================================================
# Same seed and same draw ORDER as the emcee notebook, so FLUX_DATA and PROFILE_DATA
# are bit-for-bit the same vectors. Do not reorder these two rng.normal calls.

rng = np.random.default_rng(RANDOM_SEED)

flux_model, profile_model = forward_model(TRUTH)

FLUX_SIGMA = SIGMA_FLUX_REL * flux_model
PROFILE_SIGMA = SIGMA_PROFILE_REL * profile_model.max()

FLUX_DATA = flux_model + rng.normal(0.0, FLUX_SIGMA)
PROFILE_DATA = profile_model + rng.normal(0.0, PROFILE_SIGMA, size=profile_model.shape)

print('seed                     : {}'.format(RANDOM_SEED))
print('flux    sigma (absolute) : {:.3e}  on a value of {:.5f}'.format(FLUX_SIGMA, flux_model))
print('profile sigma (absolute) : {:.3e}  on a peak of {:.5f}'.format(PROFILE_SIGMA, profile_model.max()))
print()
print('The profile is normalised to unit sum, so its {} bins carry only {} independent'
      .format(N_PROFILE_BINS, N_PROFILE_BINS - 1))
print('numbers; the overall scale lives entirely in the flux term.')

In [ ]:
# =============================================================================
# Prior transform and likelihood
# =============================================================================
# Nested sampling needs unit cube -> parameters, not a prior density.
#
#   inc        : uniform in cos(inc) over PRIOR_INC_DEG  -> inc = arccos(u)
#   v_eq       : uniform
#   lat_centre : NOT uniform. The allowed ff range shrinks towards the pole, so
#                joint-uniform (lat, ff) makes the marginal in lat proportional to
#                the allowed width w(lat). Drawn by numerical inverse CDF.
#   ff         : uniform on the slice allowed at that lat_centre.
#
# Together these give a JOINT UNIFORM prior over the allowed (lat_centre, ff) region,
# which is what the emcee notebook's rejection-style log_prior implies. Every draw is
# valid, so no likelihood evaluation is thrown away on an impossible band.

_FF_MIN, _FF_MAX = PRIOR_FF
_LAT_MIN, _LAT_MAX = PRIOR_LAT_DEG

# allowed width in ff at each latitude, on a fine grid for the inverse CDF
_LAT_GRID = np.linspace(_LAT_MIN, _LAT_MAX, 4001)
_FF_WIDTH_GRID = np.minimum(_FF_MAX, MAX_FF_AT_LAT(_LAT_GRID)) - _FF_MIN

if np.any(_FF_WIDTH_GRID <= 0.0):
    raise ValueError('ff prior is empty at some latitude in {}: the geometric ceiling '
                     'cos^2(lat) drops below PRIOR_FF[0]={}'.format(PRIOR_LAT_DEG, _FF_MIN))

# cumulative area of the allowed region, normalised to a CDF on [0, 1]
_CUM = np.concatenate([[0.0], np.cumsum(0.5 * (_FF_WIDTH_GRID[1:] + _FF_WIDTH_GRID[:-1])
                                        * np.diff(_LAT_GRID))])
_LAT_CDF = _CUM / _CUM[-1]


def prior_transform(u):
    """Map a point u in the unit 4-cube to (inc [deg], v_eq [m/s], lat_centre [deg], ff)."""
    inc_deg = np.degrees(np.arccos(u[0]))          # uniform in cos(inc) over (0, 90]

    v_eq_ms = PRIOR_V_EQ_MS[0] + u[1] * (PRIOR_V_EQ_MS[1] - PRIOR_V_EQ_MS[0])

    lat_centre_deg = np.interp(u[2], _LAT_CDF, _LAT_GRID)

    ff_max_here = min(_FF_MAX, MAX_FF_AT_LAT(lat_centre_deg))
    ff = _FF_MIN + u[3] * (ff_max_here - _FF_MIN)

    return np.array([inc_deg, v_eq_ms, lat_centre_deg, ff])


def log_likelihood(theta):
    """
    Joint chi-squared over the two observables, each with its own sigma. No covariance
    is modelled between the flux and the profile, nor between profile bins (they are
    correlated in reality through the unit-sum normalisation; ignoring that slightly
    overstates the profile's weight). Identical to the emcee notebook.

    Any exception from the forward model propagates. It is NOT caught and turned into
    -inf: a model failure inside the allowed prior volume is a bug to be seen.
    """
    flux, profile = forward_model(theta)

    chi2_flux = ((FLUX_DATA - flux) / FLUX_SIGMA) ** 2
    chi2_profile = np.sum(((PROFILE_DATA - profile) / PROFILE_SIGMA) ** 2)

    return -0.5 * (chi2_flux + chi2_profile)


print('prior_transform at cube centre : ', prior_transform(np.full(N_DIM, 0.5)))
print('log_likelihood at the truth    : {:.2f}'.format(log_likelihood(TRUTH)))
print('log_likelihood displaced       : {:.2f}'.format(log_likelihood([60.0, 5000.0, 35.0, 0.15])))

In [ ]:
# =============================================================================
# Prior transform check: does it reproduce the intended prior?
# =============================================================================
# The transform is the only genuinely new code in this notebook, so it is checked
# against brute-force rejection sampling of the same region rather than assumed.

n_check = 200000
rng_check = np.random.default_rng(RANDOM_SEED + 2)

# (a) via the transform
cube = rng_check.random((n_check, N_DIM))
draws_transform = np.array([prior_transform(c) for c in cube])

# (b) by rejection: uniform in the (lat, ff) box, keep only geometrically valid points
lat_try = rng_check.uniform(_LAT_MIN, _LAT_MAX, size=5 * n_check)
ff_try = rng_check.uniform(_FF_MIN, _FF_MAX, size=5 * n_check)
keep = ff_try < MAX_FF_AT_LAT(lat_try)
lat_reject, ff_reject = lat_try[keep], ff_try[keep]

print('rejection acceptance rate: {:.1%}  ({:,} kept)'.format(keep.mean(), keep.sum()))
print()
print('{:<20} {:>14} {:>14}'.format('quantity', 'transform', 'rejection'))
print('-' * 50)
print('{:<20} {:>14.4f} {:>14.4f}'.format('mean lat_centre', draws_transform[:, 2].mean(), lat_reject.mean()))
print('{:<20} {:>14.4f} {:>14.4f}'.format('std  lat_centre', draws_transform[:, 2].std(), lat_reject.std()))
print('{:<20} {:>14.5f} {:>14.5f}'.format('mean ff', draws_transform[:, 3].mean(), ff_reject.mean()))
print('{:<20} {:>14.5f} {:>14.5f}'.format('std  ff', draws_transform[:, 3].std(), ff_reject.std()))
print()
print('every transform draw is geometrically valid:',
      bool(np.all(draws_transform[:, 3] < MAX_FF_AT_LAT(draws_transform[:, 2]))))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(draws_transform[:, 2], bins=80, density=True, histtype='step', lw=1.5, label='transform')
axes[0].hist(lat_reject, bins=80, density=True, histtype='step', lw=1.5, ls='--', label='rejection')
axes[0].set_xlabel('lat_centre [deg]')
axes[0].set_ylabel('prior density')
axes[0].legend(fontsize=8)

axes[1].hist(draws_transform[:, 3], bins=80, density=True, histtype='step', lw=1.5, label='transform')
axes[1].hist(ff_reject, bins=80, density=True, histtype='step', lw=1.5, ls='--', label='rejection')
axes[1].set_xlabel('ff')
axes[1].legend(fontsize=8)

axes[2].scatter(draws_transform[::40, 2], draws_transform[::40, 3], s=1, alpha=0.2)
lat_line = np.linspace(_LAT_MIN, _LAT_MAX, 400)
axes[2].plot(lat_line, MAX_FF_AT_LAT(lat_line), 'r-', lw=1.5, label='cos^2(lat), geometric limit')
axes[2].plot(lat_line, np.cos(np.radians(lat_line)), 'k--', lw=1.2, label='cos(lat), emcee limit')
axes[2].plot(TRUE_LAT_CENTRE, TRUE_FF, 'r*', ms=14)
axes[2].set_xlabel('lat_centre [deg]')
axes[2].set_ylabel('ff')
axes[2].set_ylim(0, PRIOR_FF[1] * 1.05)
axes[2].legend(fontsize=8)
axes[2].set_title('shaded gap between the two curves is\nwhere healpy silently reflects the band')

fig.tight_layout()
plt.show()

## Run the sampler

Serial, so the wall time is directly comparable to the emcee notebook's serial number.
`dynesty` accepts `pool=`/`queue_size=`, but functions defined in a notebook cell do not
pickle cleanly under `multiprocessing`, so parallelising this properly means moving the
model into a module. Not done here.

In [ ]:
# =============================================================================
# Nested sampling run, timed
# =============================================================================

sampler = dynesty.NestedSampler(log_likelihood, prior_transform, N_DIM,
                                nlive=N_LIVE,
                                sample=SAMPLE_METHOD,
                                rstate=np.random.default_rng(RANDOM_SEED + 3))

t0 = time.time()
sampler.run_nested(dlogz=DLOGZ, print_progress=True)
t_nested = time.time() - t0

results = sampler.results
n_calls = int(np.sum(results.ncall))

print()
print('wall time            : {:.1f} s  ({:.2f} h)'.format(t_nested, t_nested / 3600.0))
print('iterations           : {:,}'.format(results.niter))
print('likelihood calls     : {:,}'.format(n_calls))
print('per call             : {:.4f} s   (bare forward model was {:.4f} s, so sampler '
      'overhead is {:.1%})'.format(t_nested / n_calls, T_FORWARD,
                                   (t_nested / n_calls) / T_FORWARD - 1.0))
print()
print('log Z                : {:.3f} +/- {:.3f}'.format(results.logz[-1], results.logzerr[-1]))
print()
print('log Z is the number emcee cannot give you. It is only meaningful against another')
print('model run under the SAME prior bounds -- e.g. a two-band model, or no band at all.')

In [ ]:
# =============================================================================
# Posterior: weights, recovery table, corner plot
# =============================================================================
# Nested sampling returns WEIGHTED samples. The raw sample count is not the effective
# sample size, so both are reported: a large gap means the contours are noisier than
# the sample count suggests and N_LIVE should go up.

samples = results.samples
weights = np.exp(results.logwt - results.logz[-1])
weights = weights / weights.sum()

ess = 1.0 / np.sum(weights ** 2)
samples_equal = dyfunc.resample_equal(samples, weights)

print('raw samples            : {:,}'.format(len(samples)))
print('effective sample size  : {:,.0f}  ({:.1%} of raw)'.format(ess, ess / len(samples)))
print()
print('{:<18} {:>12} {:>12} {:>12} {:>10}'.format(
    'parameter', 'truth', 'median', 'std', 'bias/std'))
print('-' * 68)

for i, label in enumerate(LABELS):
    median = dyfunc.quantile(samples[:, i], [0.5], weights=weights)[0]
    mean, cov = dyfunc.mean_and_cov(samples[:, i:i + 1], weights)
    std = float(np.sqrt(cov[0, 0]))
    bias = (median - TRUTH[i]) / std if std > 0 else np.nan
    print('{:<18} {:>12.4g} {:>12.4g} {:>12.4g} {:>10.2f}'.format(
        label, TRUTH[i], median, std, bias))

print()
print('THIS is the precision answer: the std column, provided log Z has converged')
print('(dlogz={}) and N_LIVE is high enough that ess is not the limiting factor.'.format(DLOGZ))

fig = corner.corner(samples, weights=weights, labels=LABELS, truths=TRUTH,
                    show_titles=True, title_fmt='.3g')
fig.suptitle('nested sampling, nlive={}, sample={}'.format(N_LIVE, SAMPLE_METHOD), y=1.02)
plt.show()

In [ ]:
# =============================================================================
# v_eq * sin(inc): the combination the line profile actually constrains
# =============================================================================
# v_los_map = sin(theta) * v_eq * sin(phi) * sin(inclination)  (simulator.py:500)
# so the profile width constrains v_eq*sin(inc), not v_eq. If v_eq and inc are
# individually loose but their product is tight, that is the degeneracy, not a
# sampler failure. This is the curved ridge SAMPLE_METHOD='rslice' was chosen for.

v_sini_equal = samples_equal[:, 1] * np.sin(np.radians(samples_equal[:, 0]))
v_sini_truth = TRUE_V_EQ * np.sin(np.radians(TRUE_INC))

print('{:<22} {:>12} {:>12} {:>14}'.format('quantity', 'truth', 'median', 'frac. scatter'))
print('-' * 64)
print('{:<22} {:>12.1f} {:>12.1f} {:>14.4f}'.format(
    'v_eq [m/s]', TRUE_V_EQ, np.median(samples_equal[:, 1]),
    np.std(samples_equal[:, 1]) / np.median(samples_equal[:, 1])))
print('{:<22} {:>12.1f} {:>12.1f} {:>14.4f}'.format(
    'v_eq sin(inc) [m/s]', v_sini_truth, np.median(v_sini_equal),
    np.std(v_sini_equal) / np.median(v_sini_equal)))

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(samples_equal[:, 0], samples_equal[:, 1], s=2, alpha=0.15)
inc_grid = np.linspace(*PRIOR_INC_DEG, 400)[1:]
ax.plot(inc_grid, v_sini_truth / np.sin(np.radians(inc_grid)), 'r-', lw=1.5,
        label='constant v_eq sin(inc)')
ax.plot(TRUE_INC, TRUE_V_EQ, 'r*', ms=16)
ax.set_xlim(samples_equal[:, 0].min() - 2, samples_equal[:, 0].max() + 2)
ax.set_ylim(samples_equal[:, 1].min() - 200, samples_equal[:, 1].max() + 200)
ax.set_xlabel('inc [deg]')
ax.set_ylabel('v_eq [m/s]')
ax.legend()
ax.set_title('does the posterior follow the v sin i ridge?')
fig.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Optional: save the run
# =============================================================================
# Writes to a NEW file and refuses to overwrite an existing one, per the project rule
# on not clobbering outputs.

SAVE_RUN = False    # set True to write

if SAVE_RUN:
    out_dir = './outputs/nested_sampling'
    os.makedirs(out_dir, exist_ok=True)
    out_path = '{}/snapshot_nested_seed{}_nlive{}.npz'.format(out_dir, RANDOM_SEED, N_LIVE)

    if os.path.exists(out_path):
        raise FileExistsError('{} already exists; change RANDOM_SEED or move the old file'
                              .format(out_path))

    np.savez(out_path,
             samples=samples,
             logwt=results.logwt,
             logl=results.logl,
             logz=results.logz,
             logzerr=results.logzerr,
             weights=weights,
             truth=TRUTH,
             labels=np.array(LABELS),
             seed=RANDOM_SEED,
             nlive=N_LIVE,
             sample_method=SAMPLE_METHOD,
             dlogz=DLOGZ,
             res_2d=RES_2D, res_3d=RES_3D,
             hd_ld_file=HD_LD_FILE, fac_ld_file=FAC_LD_FILE,
             line_wavelength_ang=LINE_WAVELENGTH_ANG,
             flux_data=FLUX_DATA, profile_data=PROFILE_DATA,
             t_nested=t_nested, n_calls=n_calls, t_forward=T_FORWARD)

    print('written:', out_path)
else:
    print('SAVE_RUN is False; nothing written')